# Retos · Semana 1 · El congelador del LTEE

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svillegasv/curso-genomica-filogenetica/blob/main/notebooks/semana-01/2-retos-congelador.ipynb)

**Genómica y Filogenética** · Semana 1 (U0 · Bootcamp UNIX) · Sesión 2 · Simón Villegas-Velásquez

**🎯 Resultado de aprendizaje:** RA1 · Usted se ubica en un árbol de carpetas que no conoce, lo explora y lo organiza con la shell sin guía paso a paso, y usa `--help` para descubrir opciones nuevas.<br>
**⏱️ Duración:** ≈ 1,5 h · **🧰 Requisitos:** Cuaderno guiado de la semana 1 (sesión 1)

---

## 🧬 El congelador del LTEE
Cada 75 días (unas 500 generaciones), el laboratorio de Lenski congela a −80 °C una muestra de cada una de las 12
poblaciones del experimento. Ese congelador es un registro fósil vivo: las bacterias se pueden revivir décadas
después y compararse con sus antepasados. Hoy van a explorar una **copia simulada** de ese congelador, con archivos
pequeños, y a encontrar en él algunas pistas sobre lo que pasó en la población Ara-3.

**Cómo trabajan:** en parejas, con dos roles que se turnan en cada reto. Quien **conduce** escribe los comandos;
quien **navega** lee el reto, propone el siguiente paso y revisa los mensajes de error. Solo se usan los comandos de
la sesión 1 y lo que encuentren en `--help`. No hay soluciones en este cuaderno: si se atascan, primero lean el
error, luego `--help`, luego pregunten a otra pareja y, por último, al profesor.

**Antes de empezar:** *Archivo → Guardar una copia en Drive*. Escriban aquí los nombres de la pareja:

- **Integrante 1:**
- **Integrante 2:**

In [ ]:
#@title ⚙️ Configuración (ejecute esta celda primero) { display-mode: "form" }
import os, sys, urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/svillegasv/curso-genomica-filogenetica/main"
_utils = next((p / "utils" for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "utils" / "curso.py").exists()), None)
if _utils is None:  # en Colab: se descargan las utilidades del curso
    _utils = Path(".curso/utils")
    _utils.mkdir(parents=True, exist_ok=True)
    for _f in ("curso.py", "estilo.py"):
        urllib.request.urlretrieve(f"{RAW}/utils/{_f}", _utils / _f)
sys.path.insert(0, str(_utils))

import curso
import estilo as es
import matplotlib.pyplot as plt
es.set_style()
print("Entorno listo ✔ | Colab:", curso.IN_COLAB)

## 📦 Abrir el congelador
Ejecute la celda (no hace falta mirar su código). Crea la carpeta `congelador/` en `/content`.

> Si la vuelve a ejecutar, recrea el congelador y **borra** la carpeta `proyecto/` que hayan creado.

In [ ]:
#@title 📦 Abrir el congelador { display-mode: "form" }
import random, shutil, tempfile

if curso.IN_COLAB:
    BASE = Path("/content")
else:  # pruebas automáticas o copia local
    BASE = Path(os.environ.get("CURSO_PROYECTO") or tempfile.mkdtemp(prefix="semana01-retos-"))
for carpeta in ("congelador", "proyecto"):
    shutil.rmtree(BASE / carpeta, ignore_errors=True)
C = BASE / "congelador"

GENOMA = ("AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC"
          "TTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAA"
          "TATAGGCATAGCGCACAGACAGATAAAAATTACAGAGTACACAACATCCATGAAACGCATTAGCACCACC"
          "ATTACCACCACCATCACCATTACCACAGGT")
azar = random.Random(1988)

def lecturas(n, nombre):
    bloques = []
    for i in range(1, n + 1):
        ini = azar.randrange(len(GENOMA) - 30)
        bloques.append(f"@{nombre}_{i}\n{GENOMA[ini:ini + 30]}\n+\n{'I' * 27}HGF\n")
    return "".join(bloques)

(C / "ancestro").mkdir(parents=True)
for cepa, nota in (("REL606", "Ara-"), ("REL607", "Ara+")):
    (C / "ancestro" / f"{cepa}.fasta").write_text(
        f">{cepa} ancestro de las poblaciones {nota} (fragmento simulado: 240 letras)\n"
        + "\n".join(GENOMA[i:i + 70] for i in range(0, len(GENOMA), 70)) + "\n")

GENERACIONES = [2000, 5000, 10000, 20000, 30000]
poblaciones = [f"Ara-{i}" for i in range(1, 7)] + [f"Ara+{i}" for i in range(1, 7)]
for pob in poblaciones:
    gens = GENERACIONES + ([31500, 33000] if pob == "Ara-3" else [])
    for g in gens:
        if pob == "Ara+2" and g == 20000:
            continue  # la muestra perdida (reto 2)
        d = C / pob / f"gen_{g:05d}"
        d.mkdir(parents=True)
        n = 150 if (pob == "Ara-3" and g == 20000) else azar.randint(20, 60)  # la más pesada de Ara-3 (reto 4)
        (d / "muestra.fastq").write_text(lecturas(n, f"{pob}_gen{g}"))

(C / "Ara-3" / "gen_33000" / ".nota_tecnico.txt").write_text(
    "Ara-3, generación 33 000: esta mañana el frasco amaneció mucho más turbio que los otros once.\n"
    "Hay muchas más bacterias de lo normal... ¿de qué se están alimentando, si la glucosa es tan poca?\n"
    "El medio también tiene citrato, pero E. coli no lo come cuando hay oxígeno. ¿O sí?\n")
(C / "LEAME.txt").write_text(
    "CONGELADOR DEL LTEE (copia simulada para el curso: los datos no son reales)\n\n"
    "- 12 poblaciones fundadas en 1988.\n"
    "- Ara-1 a Ara-6 vienen de la cepa REL606, que NO puede usar arabinosa (Ara-).\n"
    "- Ara+1 a Ara+6 vienen de REL607, una variante que SÍ puede usarla (Ara+) por una mutación puntual.\n"
    "  La diferencia sirve como marcador para distinguir las poblaciones en placas con arabinosa.\n"
    "- Cada 500 generaciones (75 días) se congela una muestra de cada población, a -80 °C.\n"
    "- Las carpetas gen_NNNNN guardan las lecturas de secuenciación de cada muestra.\n"
    "- Los ancestros están en ancestro/.\n")

os.chdir(BASE)
print("✔ Congelador abierto en", C)

## 🧩 Reto 1 · Reconocimiento
*≈ 10 min*

¿Cuántas poblaciones hay en el congelador y cómo se llaman? ¿Por qué unas llevan «−» y otras «+»?

> 💡 La segunda respuesta está en `LEAME.txt`. Ábralo con doble clic en el panel de archivos (📁).

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## 🧩 Reto 2 · La muestra perdida
*≈ 10 min*

En una de las poblaciones falta la muestra de una generación. ¿En cuál, y de qué generación?

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## 🧩 Reto 3 · El archivo escondido
*≈ 10 min*

Un técnico dejó una nota **oculta** en una de las carpetas de Ara-3. Encuéntrenla y léanla. ¿Qué observó y en qué
generación? ¿Qué sospecha?

> 💡 El panel de archivos no muestra los archivos ocultos. Para leer la nota, **cópienla** con un nombre que no
> empiece por punto y ábrala con doble clic.

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## 🧩 Reto 4 · La muestra más pesada
*≈ 15 min*

Busquen en `ls --help` una opción que muestre el **tamaño** de los archivos (y, si la encuentran, otra que haga
los tamaños más fáciles de leer). ¿Cuál es la muestra de Ara-3 que ocupa más espacio?

> 💡 Una muestra más pesada tiene más lecturas de secuenciación, no más mutaciones.

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## 🧩 Reto 5 · Arme su proyecto
*≈ 20 min*

Creen la carpeta `proyecto/` (en `/content`) con la estructura estándar del curso: `data/raw`, `data/processed`,
`results` y `scripts`. Después **copien** (no muevan: es el congelador del laboratorio) a `data/raw`:
- el genoma del ancestro de Ara-3, y
- tres muestras de Ara-3: la de la generación 2000, la de la 31 500 y la de la 33 000.

Todas las muestras se llaman `muestra.fastq`, así que al copiarlas pónganles un nombre que diga de dónde vienen,
sin espacios.

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## 🧩 Reto 6 · Dos caminos
*≈ 5 min*

Imaginen que están en `proyecto/data/raw`. Escriban dos rutas hacia la nota oculta del reto 3: una **relativa** y
una **absoluta**. Compruébenlas con `ls -a`.

In [ ]:
# Sus comandos aquí (una celda %%bash, o líneas con ! y %cd)

**Respuesta de la pareja:**

## ✔ Verificación del reto 5
Ejecute la celda: revisa que el proyecto tenga la forma esperada y que el congelador siga completo.

In [ ]:
#@title ✔ ¿Quedó bien el proyecto? { display-mode: "form" }
raw = BASE / "proyecto/data/raw"
def marca(ok, texto, pista=""):
    print(("✔ " if ok else "✘ ") + texto + ("" if ok else "  → " + pista))

for sub in ("data/raw", "data/processed", "results", "scripts"):
    marca((BASE / "proyecto" / sub).is_dir(), f"proyecto/{sub}", "falta esta carpeta (mkdir -p)")
archivos = [f for f in raw.iterdir() if f.is_file()] if raw.is_dir() else []
marca(any(f.name.endswith(".fasta") and "REL606" in f.name for f in archivos),
      "genoma de REL606 en data/raw", "¿cuál es el ancestro de Ara-3? Revise LEAME.txt")
fastq = [f for f in archivos if f.name.endswith(".fastq")]
marca(len(fastq) >= 3, f"{len(fastq)} muestras .fastq en data/raw", "se piden tres muestras de Ara-3")
marca(all(" " not in f.name for f in archivos), "nombres sin espacios", "renombre los archivos con espacios")
intactas = all((BASE / "congelador/Ara-3" / f"gen_{g:05d}" / "muestra.fastq").exists() for g in (2000, 31500, 33000))
marca(intactas, "el congelador sigue completo", "¿movieron en vez de copiar? Vuelva a abrir el congelador")

---

## 📓 Bitácora de la pareja
- ¿Qué aprendimos?
- ¿Qué no nos quedó claro?
- ¿Qué hacemos con esto? (¿qué comando o hábito vamos a usar de ahora en adelante?)

**Entrega:** guarde este cuaderno (con sus comandos y respuestas) en la carpeta de su pareja en la carpeta
compartida del curso.

---

<sub>**Genómica y Filogenética** · © 2026 Simón Villegas-Velásquez · Código bajo licencia MIT · Texto y figuras bajo CC BY 4.0 ·
[Repositorio](https://github.com/svillegasv/curso-genomica-filogenetica)</sub>